# TEMPLATE UTS DM II — CLUSTERING IMAGE (Google Colab)

**TWO IN ONE:** `TUGAS="gambar"` = clustering **banyak gambar** (mis. koleksi bunga/angka), setiap **gambar adalah 1 sampel**. `TUGAS="piksel"` = **segmentasi satu gambar**, setiap **piksel RGB adalah 1 sampel**.

**Cara mengisi data:** untuk banyak gambar → ZIP berisi file JPG/PNG (boleh subfolder kategori). Untuk segmentasi → satu JPG/PNG. Jalankan dulu mode **contoh** tanpa upload.

**Format ZIP yang disarankan:** `dataset/mawar/a.jpg`, `dataset/mawar/b.png`, `dataset/melati/c.jpg`; nama folder hanya label referensi opsional, tidak digunakan K-Means.

**Keyword:** `PIL Image.open convert RGB`, `numpy reshape flatten image`, `load_digits sklearn`, `KMeans cluster images`, `KMeans color quantization segmentation`.

## 0. KONFIGURASI — titik perubahan untuk UTS

In [ ]:
# [GANTI SAAT UTS]
TUGAS = "gambar"          # "gambar" (banyak gambar) / "piksel" (satu gambar RGB)
SUMBER = "contoh"         # "contoh", "upload_zip", "upload_gambar", "folder"
FOLDER_DATA = "/content/drive/MyDrive/dataset_image"  # hanya SUMBER="folder"
N_SAMPEL_GAMBAR = 300    # 100/300/500 gambar; batasi jika Colab lambat
N_SAMPEL_PIKSEL = 3000   # sampling pixel untuk training (boleh 1000 s.d. 10000)
UKURAN = 32             # semua gambar disamakan 32 x 32 agar ringan (64 lebih detail tapi lebih lambat)
K_FINAL = 3             # jumlah cluster final
RANDOM_STATE = 42
print("TUGAS:", TUGAS, "| SUMBER:", SUMBER, "| ukuran resize:", UKURAN, "| k:", K_FINAL)

**Sesudah menjalankan cell — petunjuk & interpretasi:** Pastikan `TUGAS="gambar"` bila diberikan **banyak berkas image** untuk dikelompokkan. Pastikan `TUGAS="piksel"` bila diberikan **satu gambar** untuk dipisah menjadi warna/segmen. Untuk satu gambar, set `SUMBER="upload_gambar"`. **Keyword:** `clustering image dataset vs pixel segmentation`.

## 1. Import + ingat cara membuka file

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Gambar PNG/JPG: Image.open('foto.jpg'), ubah array: np.asarray(...) ")
print("Folder: list(Path('folder').rglob('*.jpg'))")
print("Data CSV: pd.read_csv('data.csv') (jika fitur gambar sudah berupa angka)")
print("Bentuk grayscale: (tinggi, lebar); RGB: (tinggi, lebar, 3)")

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Library utama:** Pillow (`PIL`) membaca image, NumPy mengolah array, matplotlib menampilkan citra, scikit-learn melakukan K-Means/PCA. **Keyword:** `PIL Image.open shape dimensions mode`, `python pathlib rglob images`.

## 2. Load data dan sampling SEBELUM membuka banyak gambar

In [ ]:
# Data contoh: load_digits (banyak gambar 8x8) atau skimage.data.astronaut (satu RGB)
if SUMBER == "contoh":
    if TUGAS == "gambar":
        d = load_digits()
        n = min(N_SAMPEL_GAMBAR, len(d.images))
        idx = np.random.default_rng(RANDOM_STATE).choice(len(d.images), size=n, replace=False)
        images = d.images[idx]  # (n, 8, 8) grayscale, rentang intensitas 0..16
        label_ref = d.target[idx]
        sumber_gambar = [f"digit_{j}.png" for j in idx]
        print("Dataset bawaan: load_digits")
    else:
        from skimage import data
        img = data.astronaut()  # contoh satu gambar RGB
        print("Gambar bawaan: skimage.data.astronaut()")
else:
    if SUMBER in ("upload_zip", "upload_gambar"):
        from google.colab import files
        uploaded = files.upload()
        nama = next(iter(uploaded))
        if SUMBER == "upload_zip":
            with zipfile.ZipFile(nama) as z:
                z.extractall("/content/data_uts_gambar")
            root = Path("/content/data_uts_gambar")
        else:
            assert TUGAS == "piksel", "upload_gambar khusus untuk TUGAS='piksel'"
            img = np.array(Image.open(nama).convert("RGB"))
    else:
        root = Path(FOLDER_DATA)
    if SUMBER in ("upload_zip", "folder"):
        daftar_path = sorted([p for p in root.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp") and p.is_file()])
        assert len(daftar_path) > 0, "File gambar tidak ditemukan. Periksa ZIP atau FOLDER_DATA."
        if TUGAS == "gambar":
            n = min(N_SAMPEL_GAMBAR, len(daftar_path))
            idx = np.random.default_rng(RANDOM_STATE).choice(len(daftar_path), size=n, replace=False)
            sampel_path = [daftar_path[j] for j in idx]  # sampling file SEBELUM load pixel
            images = np.stack([np.asarray(Image.open(p).convert("RGB").resize((UKURAN, UKURAN))) for p in sampel_path])
            sumber_gambar = [str(p) for p in sampel_path]
            label_ref = [p.parent.name for p in sampel_path]  # hanya referensi nama folder
        else:
            img = np.asarray(Image.open(daftar_path[0]).convert("RGB"))
            print("Untuk segmentasi dipakai gambar pertama:", daftar_path[0])
if TUGAS == "gambar":
    print("Shape kumpulan gambar:", images.shape, "| jumlah gambar:", len(images))
else:
    print("Shape satu gambar:", img.shape)

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Sampling folder:** `rng.choice()` mengambil maksimal `N_SAMPEL_GAMBAR` **path** secara acak sebelum semua file dibuka, menghemat RAM. Jika kategori folder tidak seimbang, sampel global ini mengikuti proporsi yang tersedia; sampling per kelas membutuhkan modifikasi. `img.shape` dapat `(H,W,3)` sedangkan `images.shape` `(N,H,W,3)` atau `(N,H,W)`. **Keyword:** `sample random files pathlib numpy`, `PIL resize all images`.

## 3. Metadata, resolusi, dtype, channel dan visualisasi awal

In [ ]:
if TUGAS == "gambar":
    contoh = images[0]
    print("Jumlah gambar:", images.shape[0])
    print("Shape 1 gambar:", contoh.shape, "| dtype:", images.dtype)
    print("Resolusi (H x W):", contoh.shape[:2])
    print("Channel:", 1 if contoh.ndim == 2 else contoh.shape[-1])
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for ax, im in zip(axes.flat, images[:10]):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.axis("off")
    plt.suptitle("Contoh sampel gambar"); plt.tight_layout(); plt.show()
else:
    print("Resolusi asli (H x W):", img.shape[:2])
    print("Channels:", 1 if img.ndim == 2 else img.shape[2], "| dtype:", img.dtype)
    plt.figure(figsize=(6, 4)); plt.imshow(img); plt.axis("off"); plt.title("Gambar input"); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Resolusi `H×W` adalah ukuran piksel, `channels=3` berarti RGB. `dtype=uint8` lazimnya punya nilai 0–255, sedangkan `load_digits` memiliki intensitas 0–16. **Keyword:** `image.shape numpy channels RGB grayscale dtype`.

## 4. Siapkan fitur: FLATTEN (gambar) atau RESHAPE (piksel), cek shape setiap kali

In [ ]:
if TUGAS == "gambar":
    print("SEBELUM flatten:", images.shape)
    X = images.reshape(images.shape[0], -1).astype("float32")
    print("SETELAH flatten (jumlah gambar, jumlah fitur pixel):", X.shape)
    X = X / (16.0 if SUMBER == "contoh" else 255.0)
    print("SETELAH normalisasi:", X.shape, "| min/max:", round(float(X.min()), 3), round(float(X.max()), 3))
    X_scale = StandardScaler().fit_transform(X)
    print("SETELAH scaling:", X_scale.shape)
    n_komponen = min(20, X_scale.shape[0]-1, X_scale.shape[1])
    assert n_komponen >= 2, "Butuh minimal 3 gambar berbeda."
    pca = PCA(n_components=n_komponen, svd_solver="randomized", random_state=RANDOM_STATE)
    X_model = pca.fit_transform(X_scale)
    print("SETELAH PCA untuk fitur clustering:", X_model.shape)
else:
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)  # grayscale dibuat 3 channel
    print("SEBELUM flatten piksel:", img.shape)
    X_pixel = img.reshape(-1, 3).astype("float32")
    print("SETELAH flatten piksel (jumlah pixel, RGB):", X_pixel.shape)
    X_pixel = X_pixel / 255.0
    print("SETELAH normalisasi piksel:", X_pixel.shape)
    idx_pixel = np.random.default_rng(RANDOM_STATE).choice(len(X_pixel), min(N_SAMPEL_PIKSEL, len(X_pixel)), replace=False)
    X_model = X_pixel[idx_pixel]
    print("SETELAH sampling pixel (dipakai fit KMeans):", X_model.shape)

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Penting:** banyak gambar `(N,H,W,C)` → **flatten per gambar** `(N,H×W×C)`; satu gambar `(H,W,3)` → **flatten semua piksel** `(H×W,3)`. Untuk gambar banyak, PCA mengurangi dimensi agar komputasi ringan; untuk segmen warna, gunakan nilai RGB langsung agar centroid mewakili warna. **Keyword:** `numpy reshape n images flatten`, `pixel RGB clustering sklearn`.

## 5. Elbow + Silhouette untuk memilih k

In [ ]:
n_unik = len(np.unique(X_model, axis=0))
assert n_unik >= 3, "Fitur terlalu seragam: minimal 3 sampel berbeda dibutuhkan untuk analisis k."
k_candidates = list(range(2, min(7, len(X_model)-1, n_unik)+1))
inertia, sil = [], []
for k in k_candidates:
    m = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = m.fit_predict(X_model)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X_model, lab, sample_size=min(300, len(X_model)), random_state=RANDOM_STATE) if len(set(lab)) > 1 else np.nan)
print("Tabel pemilihan k:")
display(pd.DataFrame({"k": k_candidates, "inertia": inertia, "silhouette": sil}).round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_candidates, inertia, "o-"); axes[0].set(title="Elbow",xlabel="k",ylabel="Inertia")
axes[1].plot(k_candidates, sil, "o-"); axes[1].set(title="Silhouette",xlabel="k",ylabel="Skor")
plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Jumlah cluster dipilih berdasar grafik/tujuan soal. Untuk pixel, silhouette dihitung pada **sampel piksel** yang dipakai untuk fit, bukan pada keseluruhan jutaan piksel. **Keyword:** `elbow method images`, `silhouette_score sample_size`.

## 6. K-Means final + evaluasi

In [ ]:
K = min(K_FINAL, n_unik, len(X_model)-1)
model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
cluster = model.fit_predict(X_model)
print("Jumlah cluster diminta:", K, "| cluster terisi:", len(set(cluster)))
print("Distribusi cluster data training:")
display(pd.Series(cluster).value_counts().sort_index().to_frame("jumlah"))
if 1 < len(set(cluster)) < len(X_model):
    print("Silhouette:", round(silhouette_score(X_model, cluster, sample_size=min(300, len(X_model)), random_state=RANDOM_STATE), 4))
    print("Davies-Bouldin:", round(davies_bouldin_score(X_model, cluster), 4))
else:
    print("Evaluasi tidak tersedia karena hanya ada satu cluster efektif.")

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Makna unit:** mode `gambar` → distribusi menunjukkan **jumlah gambar** pada setiap cluster; mode `piksel` → distribusi menunjukkan **jumlah piksel sampling** pada setiap cluster. Label 0/1/2 bukan kelas sebenarnya. **Keyword:** `KMeans fit_predict sklearn`, `davies bouldin image clustering`.

## 7. Tampilkan hasil, proyeksi PCA atau gambar tersegmentasi

In [ ]:
if TUGAS == "gambar":
    coords = X_model[:, :2]  # 2 komponen PCA pertama
    print("Shape koordinat 2D:", coords.shape)
    plt.figure(figsize=(7, 5))
    plt.scatter(coords[:, 0], coords[:, 1], c=cluster, cmap="tab10", s=18)
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Cluster banyak gambar (PCA 2D)")
    plt.tight_layout(); plt.show()
    fig, axes = plt.subplots(1, min(K, 5), figsize=(12, 3), squeeze=False)
    for c in range(min(K, 5)):
        id_c = np.where(cluster == c)[0]
        if len(id_c):
            axes[0, c].imshow(images[id_c[0]], cmap="gray" if images[id_c[0]].ndim == 2 else None)
        axes[0, c].set_title(f"Cluster {c}\nn={len(id_c)}")
        axes[0, c].axis("off")
    plt.tight_layout(); plt.show()
    df_hasil = pd.DataFrame({"file": sumber_gambar, "cluster": cluster, "label_folder_atau_asli": label_ref})
    display(df_hasil.head(10))
    print("Crosstab referensi, jika folder merupakan kelas asli:")
    display(pd.crosstab(df_hasil["cluster"], df_hasil["label_folder_atau_asli"]))
else:
    label_semua_pixel = model.predict(X_pixel)
    print("Shape label semua piksel:", label_semua_pixel.shape)
    RGB_hasil = model.cluster_centers_[label_semua_pixel]
    print("Shape RGB hasil (flatten):", RGB_hasil.shape)
    gambar_hasil = RGB_hasil.reshape(img.shape)
    print("SETELAH reshape balik ke ukuran gambar:", gambar_hasil.shape)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(img); axes[0].set(title="Asli")
    axes[1].imshow(np.clip(gambar_hasil, 0, 1)); axes[1].set(title=f"Segmentasi warna K={K}")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Gambar banyak:** PCA menampilkan penyederhanaan, sedangkan clustering dipelajari pada ruang `X_model` (lebih banyak komponen). **Piksel:** `predict` diterapkan ke **semua piksel** setelah model dilatih pada sampel, kemudian gambar dibentuk kembali dengan `.reshape(img.shape)`. Subfolder kategori hanya evaluasi eksternal dan boleh diabaikan. **Keyword:** `KMeans color quantization predict full image`, `PCA scatter cluster images`.

## 8. Simpan output

In [ ]:
if TUGAS == "gambar":
    df_hasil.to_csv("hasil_cluster_gambar.csv", index=False)
    print("Disimpan: hasil_cluster_gambar.csv")
    display(df_hasil.head())
else:
    nama_hasil = "hasil_segmentasi.png"
    Image.fromarray((np.clip(gambar_hasil, 0, 1)*255).astype("uint8")).save(nama_hasil)
    print("Disimpan:", nama_hasil)
    display(Image.open(nama_hasil).resize((220, 220)))

**Sesudah menjalankan cell — petunjuk & interpretasi:** Untuk mode gambar hasil berupa **CSV** dengan indeks cluster tiap berkas. Untuk mode piksel hasil berupa **PNG** hasil segmentasi. File tersimpan di direktori kerja Colab. **Keyword:** `PIL Image.fromarray save`, `pandas to_csv`.

## Checklist UTS — Image

**Tugas banyak gambar:** gambar → sampling **berkas** → resize seragam → cek `(N,H,W,C)` → flatten `(N,H×W×C)` → normalisasi → PCA (opsional) → Elbow/Silhouette → K-Means → visualisasi + gambar per cluster.

**Tugas satu gambar:** gambar RGB → cek `(H,W,3)` → flatten `(H×W,3)` → sampling piksel → K-Means → `predict` semua piksel → reshape `(H,W,3)` → tampilkan hasil segmentasi.

**Peringatan:** jangan menyamakan **clustering gambar** dengan **segmentasi piksel**; unit sampelnya berbeda.